In [1]:
# =============================================================================
# Ontario Land Cover Map — MODIS-based classification
# Visualises land_cover feature from training/validation data
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import glob
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('figures', exist_ok=True)

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR       = r'C:\Users\lambe\RU_Thesis_2026\ml_data'
DOY_START_HARD = 91
DOY_END_HARD   = 310
COORD_ROUND    = 2
RANDOM_STATE   = 42

# ── Land cover class definitions ──────────────────────────────────────────────
LC_CLASSES = {
    1 : ('Temperate/sub-polar needleleaf forest',  'High'),
    2 : ('Sub-polar taiga needleleaf forest',       'High'),
    3 : ('Tropical/sub-tropical broadleaf evergr.', 'Moderate'),
    4 : ('Tropical/sub-tropical broadleaf decid.',  'Moderate'),
    5 : ('Temperate/sub-polar broadleaf deciduous', 'Moderate'),
    6 : ('Mixed forest',                            'Moderate–high'),
    7 : ('Tropical/sub-tropical shrubland',         'Moderate'),
    8 : ('Temperate/sub-polar shrubland',           'Moderate'),
    9 : ('Tropical/sub-tropical grassland',         'Moderate'),
    10: ('Temperate/sub-polar grassland',           'Moderate'),
    11: ('Sub-polar/polar shrubland-lichen-moss',   'Low–moderate'),
    12: ('Sub-polar/polar grassland-lichen-moss',   'Low'),
    13: ('Sub-polar/polar barren-lichen-moss',      'Low'),
    14: ('Wetland',                                 'Low'),
    15: ('Cropland',                                'Low–moderate'),
    16: ('Barren land',                             'Very low'),
    17: ('Urban',                                   'Human ignition'),
    18: ('Water',                                   'Non-burnable'),
    19: ('Snow/ice',                                'Non-burnable'),
}

# ── Colour palette — ecologically meaningful groupings ────────────────────────
LC_COLORS = {
    1 : '#1a5c1a',   # dark green     — needleleaf forest
    2 : '#2e7d2e',   # medium green   — taiga needleleaf
    3 : '#4caf50',   # bright green   — broadleaf evergreen
    4 : '#81c784',   # light green    — broadleaf deciduous trop
    5 : '#a5d6a7',   # pale green     — broadleaf deciduous temp
    6 : '#558b2f',   # olive green    — mixed forest
    7 : '#c8a82c',   # dark yellow    — shrubland trop
    8 : '#d4b84a',   # yellow         — shrubland temp
    9 : '#e8d44d',   # bright yellow  — grassland trop
    10: '#f0e68c',   # pale yellow    — grassland temp
    11: '#b0c4a0',   # grey-green     — polar shrubland
    12: '#c8d8b8',   # pale grey-grn  — polar grassland
    13: '#d8d8c0',   # light grey     — polar barren
    14: '#5b8fa8',   # steel blue     — wetland
    15: '#f4a460',   # sandy brown    — cropland
    16: '#c2b280',   # tan            — barren land
    17: '#e74c3c',   # red            — urban
    18: '#2196f3',   # blue           — water
    19: '#e0f0ff',   # very pale blue — snow/ice
}

# =============================================================================
# STEP 1: Load one month of data per year to get pixel land cover
# Land cover is static — one observation per pixel is sufficient
# =============================================================================
print("Loading land cover data...")

KEEP_COLS = ['longitude', 'latitude', 'land_cover', 'date']

# Load only July files from each year to keep memory manageable
# (land cover is static so any month works)
sample_dfs = []
for year in list(range(2010, 2023)):
    split    = 'train' if year <= 2020 else 'val'
    # Try July first, then any available month
    patterns = [
        os.path.join(DATA_DIR, f'ML_{split}_{year}07.csv'),
        os.path.join(DATA_DIR, f'ML_{split}_{year}06.csv'),
        os.path.join(DATA_DIR, f'ML_{split}_{year}08.csv'),
    ]
    loaded = False
    for pattern in patterns:
        files = glob.glob(pattern)
        if files:
            try:
                df = pd.read_csv(
                    files[0], usecols=KEEP_COLS,
                    on_bad_lines='skip')
                sample_dfs.append(df)
                print(f"  {year}: loaded {os.path.basename(files[0])} "
                      f"({len(df):,} rows)")
                loaded = True
                break
            except Exception as e:
                print(f"  WARNING {year}: {e}")
    if not loaded:
        print(f"  WARNING: no file found for {year}")

df_raw = pd.concat(sample_dfs, ignore_index=True)
df_raw['land_cover'] = df_raw['land_cover'].astype(int)

# Round to pixel grid and keep one observation per pixel
df_raw['lon_r'] = df_raw['longitude'].round(COORD_ROUND)
df_raw['lat_r'] = df_raw['latitude'].round(COORD_ROUND)

# One row per pixel — take mode of land cover across years
# in case of slight variation
df_pixels = (
    df_raw
    .groupby(['lon_r', 'lat_r'])['land_cover']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
)

print(f"\n  Unique pixels: {len(df_pixels):,}")
print(f"\n  Land cover distribution:")
for code in sorted(df_pixels['land_cover'].unique()):
    n   = (df_pixels['land_cover'] == code).sum()
    pct = n / len(df_pixels) * 100
    name = LC_CLASSES.get(code, ('Unknown', ''))[0]
    print(f"    Class {code:>2}: {name:<45} "
          f"{n:>6,}  ({pct:.1f}%)")

# =============================================================================
# STEP 2: Compute dot size
# =============================================================================
lon_range = df_pixels['lon_r'].max() - df_pixels['lon_r'].min()
lat_range = df_pixels['lat_r'].max() - df_pixels['lat_r'].min()

FIG_W, FIG_H   = 18, 13
DPI            = 300
AXES_W         = FIG_W * 0.72
AXES_H         = FIG_H * 0.80
PIXEL_STEP     = 0.09

PX_PER_DEG_X  = (AXES_W * DPI) / lon_range
PX_PER_DEG_Y  = (AXES_H * DPI) / lat_range
PIXEL_PX      = PIXEL_STEP * min(PX_PER_DEG_X, PX_PER_DEG_Y)
POINTS_PER_PX = DPI / 72
S_BASE        = (PIXEL_PX / POINTS_PER_PX) ** 2 * 0.72

print(f"\n  Dot size (s): {S_BASE:.1f}")

# =============================================================================
# STEP 3: Plot
# =============================================================================
print("\nGenerating land cover map...")

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
fig.patch.set_facecolor('white')
fig.subplots_adjust(right=0.72)   # leave room for legend

# Plot each land cover class separately for legend control
present_classes = sorted(df_pixels['land_cover'].unique())

for lc_code in present_classes:
    subset = df_pixels[df_pixels['land_cover'] == lc_code]
    color  = LC_COLORS.get(lc_code, '#999999')

    ax.scatter(
        subset['lon_r'], subset['lat_r'],
        c          = color,
        s          = S_BASE * 0.90,
        alpha      = 0.85,
        linewidths = 0,
        rasterized = True,
        zorder     = 2 + (lc_code * 0.1),
        marker     = 's'
    )

# ── Axis limits ────────────────────────────────────────────────────────────────
ax.set_xlim(df_pixels['lon_r'].min() - 0.5,
            df_pixels['lon_r'].max() + 0.5)
ax.set_ylim(df_pixels['lat_r'].min() - 0.3,
            df_pixels['lat_r'].max() + 0.3)

# ── Legend ─────────────────────────────────────────────────────────────────────
legend_patches = []
for lc_code in present_classes:
    if lc_code not in LC_CLASSES:
        continue
    name, fuel = LC_CLASSES[lc_code]
    color      = LC_COLORS.get(lc_code, '#999999')
    n          = (df_pixels['land_cover'] == lc_code).sum()
    legend_patches.append(mpatches.Patch(
        facecolor = color,
        edgecolor = '#555555',
        linewidth = 0.3,
        label     = f'{lc_code:>2}. {name}  [{fuel}]  '
                    f'(n={n:,})'
    ))

# Place legend outside the plot to the right
legend = ax.legend(
    handles       = legend_patches,
    title         = 'Land Cover Class  [Fuel load]',
    title_fontsize= 9,
    fontsize      = 7.5,
    loc           = 'upper left',
    bbox_to_anchor= (1.01, 1.0),
    framealpha    = 0.97,
    edgecolor     = '#cccccc',
    borderpad     = 0.8,
    labelspacing  = 0.5,
)
legend.get_title().set_fontweight('bold')

# ── Labels ─────────────────────────────────────────────────────────────────────
ax.set_xlabel('Longitude (°W)', fontsize=11)
ax.set_ylabel('Latitude (°N)',  fontsize=11)
ax.set_title(
    'Ontario Land Cover Classification\n'
    'MODIS-based  |  Study domain: 2010–2022',
    fontsize=13, fontweight='bold', pad=14
)
ax.tick_params(labelsize=10)
ax.grid(color='#e8e8e8', linewidth=0.4, alpha=0.6, zorder=1)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)

# ── Stats annotation ────────────────────────────────────────────────────────────
# Top 5 classes by pixel count
top5 = (
    df_pixels['land_cover']
    .value_counts()
    .head(5)
)
stats_lines = [
    f"Total pixels : {len(df_pixels):,}",
    f"Classes      : {len(present_classes)}",
    "─" * 26,
    "Top 5 classes:",
]
for code, count in top5.items():
    name = LC_CLASSES.get(code, ('Unknown',))[0]
    short_name = name[:22] + '…' if len(name) > 22 else name
    stats_lines.append(
        f"  {code:>2}. {short_name:<24} {count:>6,}")

ax.text(
    0.015, 0.015,
    '\n'.join(stats_lines),
    transform           = ax.transAxes,
    fontsize            = 7.5,
    verticalalignment   = 'bottom',
    horizontalalignment = 'left',
    fontfamily          = 'monospace',
    bbox                = dict(boxstyle='round,pad=0.6',
                               facecolor='white',
                               edgecolor='#cccccc',
                               alpha=0.95)
)

fig.text(
    0.38, -0.01,
    'Source: MODIS Land Cover (MCD12Q1)  |  '
    'Processed via Google Earth Engine  |  '
    'Study domain: Ontario, Canada',
    ha='center', fontsize=8,
    color='#888888', style='italic'
)

fig.savefig('figures/fig_ontario_land_cover_map.png',
            dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig('figures/fig_ontario_land_cover_map.pdf',
            bbox_inches='tight', facecolor='white')

print("  Saved: figures/fig_ontario_land_cover_map.png")
print("  Saved: figures/fig_ontario_land_cover_map.pdf")
plt.close(fig)

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("LAND COVER MAP — SUMMARY")
print(f"{'='*60}")
print(f"  Total pixels   : {len(df_pixels):,}")
print(f"  Classes present: {len(present_classes)}")
print(f"\n  Full distribution:")
print(f"  {'Code':>4}  {'Class name':<45} {'Pixels':>8}  {'%':>6}")
print(f"  {'─'*68}")
for code in sorted(present_classes):
    n    = (df_pixels['land_cover'] == code).sum()
    pct  = n / len(df_pixels) * 100
    name = LC_CLASSES.get(code, ('Unknown',))[0]
    print(f"  {code:>4}  {name:<45} {n:>8,}  {pct:>5.1f}%")

print(f"\nLaTeX:")
print(f"  \\includegraphics[width=\\textwidth]"
      f"{{figures/fig_ontario_land_cover_map}}")

Loading land cover data...
  2010: loaded ML_train_201007.csv (367,846 rows)
  2011: loaded ML_train_201107.csv (367,846 rows)
  2012: loaded ML_train_201207.csv (367,846 rows)
  2013: loaded ML_train_201307.csv (367,846 rows)
  2014: loaded ML_train_201407.csv (367,846 rows)
  2015: loaded ML_train_201507.csv (367,846 rows)
  2016: loaded ML_train_201607.csv (367,846 rows)
  2017: loaded ML_train_201707.csv (367,846 rows)
  2018: loaded ML_train_201807.csv (367,846 rows)
  2019: loaded ML_train_201907.csv (367,846 rows)
  2020: loaded ML_train_202007.csv (367,846 rows)
  2021: loaded ML_val_202107.csv (367,846 rows)
  2022: loaded ML_val_202207.csv (367,846 rows)

  Unique pixels: 11,866

  Land cover distribution:
    Class  1: Temperate/sub-polar needleleaf forest          5,199  (43.8%)
    Class  5: Temperate/sub-polar broadleaf deciduous          279  (2.4%)
    Class  6: Mixed forest                                   2,359  (19.9%)
    Class  8: Temperate/sub-polar shrubland    